In [ ]:
"""
Nexus Datacom - Power Monitoring Dashboard Generator
Author: Alexander Korshunov
Purpose: Generate interactive Plotly dashboards from SQLite database

Usage: python generate_dashboards.py
"""

import sqlite3
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
import os

# ============================================
# 1. DATABASE SETUP & DATA GENERATION
# ============================================

DB_PATH = 'nexus_datacom.db'

def create_database():
    """
    Create SQLite database with synthetic data for 25 data centers and 150+ halls.
    Generates realistic consumption and commercial data.
    """

    # Remove existing database if it exists
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # Create tables
    cursor.execute('''
        CREATE TABLE data_centers (
            dc_id INTEGER PRIMARY KEY AUTOINCREMENT,
            dc_name TEXT UNIQUE NOT NULL,
            dc_type TEXT NOT NULL,
            total_halls INTEGER NOT NULL,
            base_power INTEGER NOT NULL
        )
    ''')

    cursor.execute('''
        CREATE TABLE halls (
            hall_id INTEGER PRIMARY KEY AUTOINCREMENT,
            dc_id INTEGER NOT NULL,
            hall_name TEXT NOT NULL,
            project_power INTEGER NOT NULL,
            FOREIGN KEY (dc_id) REFERENCES data_centers (dc_id)
        )
    ''')

    cursor.execute('''
        CREATE TABLE consumption (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            hall_id INTEGER NOT NULL,
            fact_power INTEGER NOT NULL,
            record_date DATE NOT NULL,
            FOREIGN KEY (hall_id) REFERENCES halls (hall_id)
        )
    ''')

    cursor.execute('''
        CREATE TABLE commercial (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            hall_id INTEGER NOT NULL,
            comm_power INTEGER NOT NULL,
            record_date DATE NOT NULL,
            FOREIGN KEY (hall_id) REFERENCES halls (hall_id)
        )
    ''')

    # Define data centers
    data_centers = [
        # New DCs (low utilization)
        ('Apex-3', 'new', 5, 950),
        ('Apex-4', 'new', 8, 1750),
        # Medium DCs
        ('Apex-1', 'medium', 8, 1850),
        ('Apex-2', 'medium', 6, 2100),
        # Moderate DCs (45-55% load, 110-120% sales)
        ('Core-1', 'moderate', 4, 1450),
        ('Core-2', 'moderate', 5, 1280),
        ('Core-3', 'moderate', 4, 1180),
        ('Core-4', 'moderate', 6, 1620),
        ('Cascade-1', 'moderate', 3, 720),
        ('Cascade-2', 'moderate', 4, 890),
        ('Cascade-3', 'moderate', 3, 660),
        ('Cascade-4', 'moderate', 6, 1050),
        ('Cascade-5', 'moderate', 3, 770),
        ('Cascade-6', 'moderate', 4, 920),
        ('Gateway-1', 'moderate', 3, 630),
        ('Gateway-2', 'moderate', 4, 580),
        ('Harbor-1', 'moderate', 3, 820),
        ('Harbor-2', 'moderate', 4, 760),
        # Legacy DCs (high utilization)
        ('Delta-1', 'legacy', 3, 540),
        ('Delta-2', 'legacy', 3, 490),
        ('Delta-3', 'legacy', 3, 620),
        ('Eclipse-1', 'legacy', 3, 380),
        ('Eclipse-2', 'legacy', 4, 440),
        ('Forge-1', 'legacy', 3, 1120),
        ('Forge-3', 'legacy', 3, 860),
        ('Forge-4', 'legacy', 4, 720),
        # Unique overloaded DC
        ('Forge-2', 'critical', 5, 980),
    ]

    # Insert data centers
    for dc in data_centers:
        cursor.execute('''
            INSERT INTO data_centers (dc_name, dc_type, total_halls, base_power)
            VALUES (?, ?, ?, ?)
        ''', dc)

    conn.commit()

    # Get DC IDs and generate halls
    cursor.execute('SELECT dc_id, dc_name, dc_type, base_power FROM data_centers')
    dcs = cursor.fetchall()

    hall_data = []

    for dc_id, dc_name, dc_type, base_power in dcs:
        # Number of halls varies by DC type
        if dc_type == 'new':
            halls_count = 5 if 'Apex-3' in dc_name else 8
            load_range = (0.15, 0.35)      # 15-35% utilization
            sales_range = (0.25, 0.45)      # 25-45% sold
        elif dc_type == 'medium':
            halls_count = 8 if 'Apex-1' in dc_name else 6
            load_range = (0.50, 0.75)       # 50-75% utilization
            sales_range = (0.75, 1.00)      # 75-100% sold
        elif dc_type == 'moderate':
            halls_count = 3 if 'Cascade-1' in dc_name else (6 if 'Cascade-4' in dc_name else 4)
            load_range = (0.45, 0.55)       # 45-55% utilization
            sales_range = (1.10, 1.20)      # 110-120% sold (oversold)
        elif dc_type == 'legacy':
            halls_count = 3
            load_range = (0.70, 0.90)       # 70-90% utilization
            sales_range = (1.15, 1.35)      # 115-135% sold
        else:  # critical (Forge-2)
            halls_count = 5
            load_range = (0.85, 2.20)       # 85-220% utilization (some extremely overloaded)
            sales_range = (1.70, 2.20)      # 170-220% sold

        for hall_num in range(1, halls_count + 1):
            # Generate project power with variation
            import random
            variation = random.uniform(-0.3, 0.5)
            project_power = int(base_power * (1 + variation))

            # Round to nice numbers
            if project_power < 100:
                project_power = round(project_power / 10) * 10
            else:
                project_power = round(project_power / 50) * 50

            # Generate fact_power based on load_range
            if dc_name == 'Forge-2' and hall_num in [2, 3, 5]:
                # Extreme overload for specific halls
                load_factor = random.uniform(1.80, 2.20)
            else:
                load_factor = random.uniform(load_range[0], load_range[1])

            fact_power = int(project_power * load_factor)

            # Generate comm_power based on sales_range
            if dc_name == 'Forge-2' and hall_num in [2, 3, 5]:
                # Extreme overselling for specific halls
                sale_factor = random.uniform(2.00, 2.40)
            else:
                sale_factor = random.uniform(sales_range[0], sales_range[1])

            comm_power = int(project_power * sale_factor)

            # Insert hall
            cursor.execute('''
                INSERT INTO halls (dc_id, hall_name, project_power)
                VALUES (?, ?, ?)
            ''', (dc_id, f'Hall-{hall_num:02d}', project_power))

            hall_id = cursor.lastrowid

            # Store for later use
            hall_data.append({
                'hall_id': hall_id,
                'dc_name': dc_name,
                'hall_name': f'Hall-{hall_num:02d}',
                'project_power': project_power,
                'fact_power': fact_power,
                'comm_power': comm_power
            })

    conn.commit()

    # Insert consumption and commercial data
    current_date = datetime.now().strftime('%Y-%m-%d')

    for hall in hall_data:
        cursor.execute('''
            INSERT INTO consumption (hall_id, fact_power, record_date)
            VALUES (?, ?, ?)
        ''', (hall['hall_id'], hall['fact_power'], current_date))

        cursor.execute('''
            INSERT INTO commercial (hall_id, comm_power, record_date)
            VALUES (?, ?, ?)
        ''', (hall['hall_id'], hall['comm_power'], current_date))

    conn.commit()
    conn.close()

    print(f"✅ Database created: {DB_PATH}")
    print(f"   - Data centers: {len(data_centers)}")
    print(f"   - Halls: {len(hall_data)}")

    return hall_data


# ============================================
# 2. DATA LOADING FROM SQL
# ============================================

def load_consumption_data():
    """
    Load consumption data from SQLite database.
    Returns DataFrame with dc, hall, project_power, fact_power, load_percent.
    """
    conn = sqlite3.connect(DB_PATH)

    query = '''
        SELECT
            dc.dc_name as dc,
            h.hall_name as hall,
            h.project_power,
            c.fact_power,
            CAST(c.fact_power AS FLOAT) / h.project_power * 100 as load_percent
        FROM halls h
        JOIN data_centers dc ON h.dc_id = dc.dc_id
        JOIN consumption c ON h.hall_id = c.hall_id
        WHERE c.record_date = (SELECT MAX(record_date) FROM consumption)
        ORDER BY dc.dc_name, h.hall_name
    '''

    df = pd.read_sql(query, conn)
    conn.close()
    return df


def load_commercial_data():
    """
    Load commercial data from SQLite database.
    Returns DataFrame with dc, hall, project_power, comm_power, sale_percent, risk_level.
    """
    conn = sqlite3.connect(DB_PATH)

    query = '''
        SELECT
            dc.dc_name as dc,
            h.hall_name as hall,
            h.project_power,
            cm.comm_power,
            CAST(cm.comm_power AS FLOAT) / h.project_power * 100 as sale_percent
        FROM halls h
        JOIN data_centers dc ON h.dc_id = dc.dc_id
        JOIN commercial cm ON h.hall_id = cm.hall_id
        WHERE cm.record_date = (SELECT MAX(record_date) FROM commercial)
        ORDER BY dc.dc_name, h.hall_name
    '''

    df = pd.read_sql(query, conn)
    conn.close()

    # Add risk level
    def get_risk_level(percent):
        if percent < 100:
            return "Low"
        elif percent < 120:
            return "Optimal"
        elif percent < 140:
            return "Elevated"
        elif percent < 160:
            return "High"
        else:
            return "Critical"

    df['risk_level'] = df['sale_percent'].apply(get_risk_level)

    # Create unique key for each row (important for later dict lookups)
    df['unique_key'] = df['dc'] + '|' + df['hall']

    return df


def get_dc_aggregated():
    """
    Get aggregated data by data center from SQL.
    """
    conn = sqlite3.connect(DB_PATH)

    query = '''
        SELECT
            dc.dc_name as dc,
            SUM(h.project_power) as total_project_power,
            SUM(c.fact_power) as total_fact_power,
            SUM(cm.comm_power) as total_comm_power,
            CAST(SUM(c.fact_power) AS FLOAT) / SUM(h.project_power) * 100 as dc_load_percent,
            CAST(SUM(cm.comm_power) AS FLOAT) / SUM(h.project_power) * 100 as dc_sale_percent
        FROM data_centers dc
        JOIN halls h ON dc.dc_id = h.dc_id
        JOIN consumption c ON h.hall_id = c.hall_id
        JOIN commercial cm ON h.hall_id = cm.hall_id
        WHERE c.record_date = (SELECT MAX(record_date) FROM consumption)
        GROUP BY dc.dc_name
        ORDER BY dc.dc_name
    '''

    df = pd.read_sql(query, conn)
    conn.close()
    return df


# ============================================
# 3. COLOR CALCULATIONS
# ============================================

def calculate_consumption_color(percent):
    """
    Calculate RGB color based on load percentage for consumption dashboard.
    Green (0-60%) -> Yellow (60-80%) -> Orange (80-100%) -> Red (100%+)
    """
    if percent < 60:
        # Green to Yellow gradient
        pos = percent / 60.0
        r1, g1, b1 = 0x1a, 0x98, 0x50  # Green
        r2, g2, b2 = 0xfe, 0xe0, 0x8b  # Yellow
    elif percent < 80:
        # Yellow to Orange gradient
        pos = (percent - 60) / 20.0
        r1, g1, b1 = 0xfe, 0xe0, 0x8b  # Yellow
        r2, g2, b2 = 0xfd, 0xae, 0x61  # Orange
    elif percent < 100:
        # Orange to Red gradient
        pos = (percent - 80) / 20.0
        r1, g1, b1 = 0xfd, 0xae, 0x61  # Orange
        r2, g2, b2 = 0xd7, 0x30, 0x27  # Red
    else:
        # 100%+ = Solid Red
        return 'rgb(215, 48, 39)'

    r = int(r1 + (r2 - r1) * pos)
    g = int(g1 + (g2 - g1) * pos)
    b = int(b1 + (b2 - b1) * pos)

    return f'rgb({r}, {g}, {b})'


def calculate_sales_color(percent):
    """
    Calculate RGB color based on sale percentage for commercial dashboard.
    Light Green (<80%) -> Green (80-100%) -> Dark Green (100-120%) ->
    Yellow (120-140%) -> Orange (140-160%) -> Red (160%+)
    """
    if percent < 80:
        # Light Green to Medium Green
        pos = percent / 80.0
        r1, g1, b1 = 0xdc, 0xed, 0xc8  # Light Green
        r2, g2, b2 = 0x91, 0xcf, 0x60  # Medium Green
    elif percent < 100:
        # Medium Green to Dark Green
        pos = (percent - 80) / 20.0
        r1, g1, b1 = 0x91, 0xcf, 0x60  # Medium Green
        r2, g2, b2 = 0x1a, 0x98, 0x50  # Dark Green
    elif percent < 120:
        # Dark Green to Green-Yellow (optimal zone)
        pos = (percent - 100) / 20.0
        r1, g1, b1 = 0x1a, 0x98, 0x50  # Dark Green
        r2, g2, b2 = 0x78, 0xb7, 0x2e  # Green-Yellow
    elif percent < 140:
        # Yellow to Orange (warning zone)
        pos = (percent - 120) / 20.0
        r1, g1, b1 = 0xfe, 0xe0, 0x8b  # Yellow
        r2, g2, b2 = 0xfd, 0xae, 0x61  # Orange
    elif percent < 160:
        # Orange to Red (high risk zone)
        pos = (percent - 140) / 20.0
        r1, g1, b1 = 0xfd, 0xae, 0x61  # Orange
        r2, g2, b2 = 0xd7, 0x30, 0x27  # Red
    else:
        # Deep Red for critical
        pos = min((percent - 160) / 40.0, 1.0)
        r1, g1, b1 = 0xd7, 0x30, 0x27  # Red
        r2, g2, b2 = 0x8b, 0x00, 0x00  # Dark Red
        r = int(r1 + (r2 - r1) * pos)
        g = int(g1 + (g2 - g1) * pos)
        b = int(b1 + (b2 - b1) * pos)
        return f'rgb({r}, {g}, {b})'

    r = int(r1 + (r2 - r1) * pos)
    g = int(g1 + (g2 - g1) * pos)
    b = int(b1 + (b2 - b1) * pos)

    return f'rgb({r}, {g}, {b})'


# ============================================
# 4. DASHBOARD GENERATION
# ============================================

def generate_consumption_dashboard(df_consumption, df_dc_aggregated):
    """
    Generate power consumption treemap dashboard.
    """
    # Add colors to dataframe
    df_consumption['color'] = df_consumption['load_percent'].apply(calculate_consumption_color)

    # Create unique key for each hall (dc + hall)
    df_consumption['unique_key'] = df_consumption['dc'] + '|' + df_consumption['hall']

    # Create treemap
    fig = px.treemap(
        df_consumption,
        path=['dc', 'hall'],
        values='project_power',
        hover_data=['project_power', 'fact_power', 'load_percent']
    )

    # Prepare custom data using unique key instead of just hall name
    hall_data_dict = {}
    for _, row in df_consumption.iterrows():
        unique_key = row['unique_key']
        hall_data_dict[unique_key] = {
            'fact_power': row['fact_power'],
            'load_percent': row['load_percent'],
            'color': row['color'],
            'dc': row['dc']
        }

    dc_data_dict = df_dc_aggregated.set_index('dc')[['total_fact_power', 'dc_load_percent']].to_dict('index')

    # Build colors and text
    colors_list = []
    text_list = []
    customdata_list = []

    for i, label in enumerate(fig.data[0].labels):
        parent = fig.data[0].parents[i]

        if parent == "":  # This is a data center (root node)
            dc_name = label
            colors_list.append('rgb(220, 220, 220)')  # Gray for DCs

            if dc_name in dc_data_dict:
                fact = dc_data_dict[dc_name]['total_fact_power']
                percent = dc_data_dict[dc_name]['dc_load_percent']
                customdata_list.append([f'{fact:.1f}', f'{percent:.1f}%'])
                text_list.append(f'{dc_name}<br>{percent:.1f}%')
            else:
                customdata_list.append(['--', '--'])
                text_list.append(label)
        else:  # This is a hall - need to find it by dc+hall
            dc_name = parent
            unique_key = f"{dc_name}|{label}"

            if unique_key in hall_data_dict:
                hall_info = hall_data_dict[unique_key]
                colors_list.append(hall_info['color'])
                percent = hall_info['load_percent']
                fact = hall_info['fact_power']
                text_list.append(f'{label}<br>{percent:.1f}%')
                customdata_list.append([f'{fact:.1f}', f'{percent:.1f}%'])
            else:
                colors_list.append('lightgrey')
                text_list.append(label)
                customdata_list.append(['--', '--'])

    # Update trace with all customizations
    fig.update_traces(
        marker=dict(colors=colors_list),
        textinfo='text+value',
        text=text_list,
        customdata=customdata_list,
        texttemplate='<b>%{text}</b><br>%{value:.0f} kW',
        hovertemplate='<b>%{label}</b><br>'
                      'Projected Power: %{value:.0f} kW<br>'
                      'Actual Power: %{customdata[0]} kW<br>'
                      'Load: <b>%{customdata[1]}</b><br>'
                      '<extra></extra>',
        textfont=dict(size=14),
        marker_line=dict(width=2, color='white')
    )

    # Layout
    fig.update_layout(
        title=dict(
            text='<b>Nexus Datacom - Power Consumption Heatmap</b><br>'
                 '<span style="font-size:14px">Block size = Projected Power | Color = Load %</span>',
            x=0.5,
            xanchor='center'
        ),
        margin=dict(t=100, l=5, r=5, b=5),
        width=1600,
        height=900,
        showlegend=False
    )

    # Add color legend annotation
    fig.add_annotation(
        x=1.3, y=0.98,
        xref='paper', yref='paper',
        text='<b>🎯 LOAD LEVELS:</b><br>'
             '<span style="color:#1a9850">▉ 0-60%: Normal</span><br>'
             '<span style="color:#fee08b">▉ 60-80%: Attention</span><br>'
             '<span style="color:#fdae61">▉ 80-100%: High Load</span><br>'
             '<span style="color:#d73027">▉ 100%+: Overload</span><br><br>'
             '<b>📦 Data Centers:</b><br>'
             '<span style="color:rgb(220,220,220)">▉ Gray boxes</span>',
        showarrow=False,
        align='left',
        bgcolor='rgba(255,255,255,0.95)',
        bordercolor='black',
        borderwidth=1,
        borderpad=8,
        font=dict(size=11)
    )

    fig.write_html("heatmap_consumption.html", include_plotlyjs='cdn')
    print("✅ Dashboard saved: heatmap_consumption.html")

    return fig


def generate_sales_dashboard(df_commercial, df_dc_aggregated):
    """
    Generate commercial sales risk treemap dashboard.
    """
    # Add colors to dataframe
    df_commercial['color'] = df_commercial['sale_percent'].apply(calculate_sales_color)

    # Create unique key for each hall (dc + hall)
    df_commercial['unique_key'] = df_commercial['dc'] + '|' + df_commercial['hall']

    # Create treemap
    fig = px.treemap(
        df_commercial,
        path=['dc', 'hall'],
        values='project_power',
        hover_data=['project_power', 'comm_power', 'sale_percent', 'risk_level']
    )

    # Prepare custom data using unique key
    hall_data_dict = {}
    for _, row in df_commercial.iterrows():
        unique_key = row['unique_key']
        hall_data_dict[unique_key] = {
            'comm_power': row['comm_power'],
            'sale_percent': row['sale_percent'],
            'risk_level': row['risk_level'],
            'color': row['color'],
            'dc': row['dc']
        }

    dc_data_dict = df_dc_aggregated.set_index('dc')[['total_comm_power', 'dc_sale_percent']].to_dict('index')

    # Build colors and text
    colors_list = []
    text_list = []
    customdata_list = []

    for i, label in enumerate(fig.data[0].labels):
        parent = fig.data[0].parents[i]

        if parent == "":  # Data center
            dc_name = label
            colors_list.append('rgb(220, 220, 220)')

            if dc_name in dc_data_dict:
                comm = dc_data_dict[dc_name]['total_comm_power']
                percent = dc_data_dict[dc_name]['dc_sale_percent']
                customdata_list.append([f'{comm:.1f}', f'{percent:.1f}%', ''])

                # Add warning symbol for high risk DCs
                warning = " ⚠️" if percent >= 160 else (" ⚠" if percent >= 140 else "")
                text_list.append(f'{dc_name}<br>{percent:.1f}%{warning}')
            else:
                customdata_list.append(['--', '--', ''])
                text_list.append(label)
        else:  # Hall - find by dc+hall
            dc_name = parent
            unique_key = f"{dc_name}|{label}"

            if unique_key in hall_data_dict:
                hall_info = hall_data_dict[unique_key]
                colors_list.append(hall_info['color'])
                percent = hall_info['sale_percent']
                comm = hall_info['comm_power']
                risk = hall_info['risk_level']

                # Add warning symbol
                warning = " ⚠️" if percent >= 160 else (" ⚠" if percent >= 140 else "")
                text_list.append(f'{label}<br>{percent:.0f}%{warning}')
                customdata_list.append([f'{comm:.1f}', f'{percent:.1f}%', risk])
            else:
                colors_list.append('lightgrey')
                text_list.append(label)
                customdata_list.append(['--', '--', ''])

    fig.update_traces(
        marker=dict(colors=colors_list),
        textinfo='text+value',
        text=text_list,
        customdata=customdata_list,
        texttemplate='<b>%{text}</b><br>%{value:.0f} kW',
        hovertemplate='<b>%{label}</b><br>'
                      'Projected Power: %{value:.0f} kW<br>'
                      'Commercial Power: %{customdata[0]} kW<br>'
                      'Sold: <b>%{customdata[1]}</b><br>'
                      'Risk Level: %{customdata[2]}<br>'
                      '<extra></extra>',
        textfont=dict(size=14),
        marker_line=dict(width=2, color='white')
    )

    # Layout with title and legend
    fig.update_layout(
        title=dict(
            text='<b>Nexus Datacom - Sales Risk Heatmap</b><br>'
                 '<span style="font-size:14px">Block size = Projected Power | Color = Risk Level (120% is industry norm)</span>',
            x=0.5,
            xanchor='center'
        ),
        margin=dict(t=100, l=5, r=5, b=5),
        width=1600,
        height=900,
        showlegend=False
    )

    # Add risk legend
    fig.add_annotation(
        x=1.3, y=0.98,
        xref='paper', yref='paper',
        text='<b>🎯 RISK LEVELS:</b><br>'
             '<span style="color:#91cf60">▉ &lt;100%: Low Risk</span><br>'
             '<span style="color:#1a9850">▉ 100-120%: Optimal ✓</span><br>'
             '<span style="color:#fee08b">▉ 120-140%: Warning ⚠</span><br>'
             '<span style="color:#fdae61">▉ 140-160%: High Risk ⚠️</span><br>'
             '<span style="color:#d73027">▉ 160%+: Critical ⚠️</span><br><br>'
             '<b>📊 INTERPRETATION:</b><br>'
             '• <b>120%</b> is industry oversubscription norm<br>'
             '• >140% requires load analysis<br>'
             '• >160% critical overload risk<br><br>'
             '<b>📦 Data Centers:</b><br>'
             '<span style="color:rgb(220,220,220)">▉ Gray boxes</span>',
        showarrow=False,
        align='left',
        bgcolor='rgba(255,255,255,0.95)',
        bordercolor='black',
        borderwidth=1,
        borderpad=8,
        font=dict(size=11)
    )

    fig.write_html("heatmap_sales.html", include_plotlyjs='cdn')
    print("✅ Dashboard saved: heatmap_sales.html")

    return fig


# ============================================
# 5. MAIN EXECUTION
# ============================================

def main():
    """Main execution function - generates database and dashboards"""

    print("\n" + "=" * 60)
    print("NEXUS DATACOM - POWER MONITORING DASHBOARDS")
    print("=" * 60 + "\n")

    # Step 1: Create database with synthetic data
    print("📦 Step 1: Creating SQLite database...")
    hall_data = create_database()

    # Step 2: Load data from SQL
    print("\n📊 Step 2: Loading data from database...")
    df_consumption = load_consumption_data()
    df_commercial = load_commercial_data()
    df_dc_aggregated = get_dc_aggregated()

    # Print summary statistics
    print(f"\n📈 Data Summary:")
    print(f"   - Total halls: {len(df_consumption)}")
    print(f"   - Total projected power: {df_consumption['project_power'].sum():,.0f} kW")
    print(f"   - Average load: {df_consumption['load_percent'].mean():.1f}%")
    print(f"   - Average sales: {df_commercial['sale_percent'].mean():.1f}%")

    # Step 3: Generate dashboards
    print("\n🎨 Step 3: Generating interactive dashboards...")
    generate_consumption_dashboard(df_consumption, df_dc_aggregated)
    generate_sales_dashboard(df_commercial, df_dc_aggregated)

    print("\n" + "=" * 60)
    print("✅ ALL DASHBOARDS GENERATED SUCCESSFULLY!")
    print("=" * 60)
    print("\n📁 Output files:")
    print("   - heatmap_consumption.html (Power consumption dashboard)")
    print("   - heatmap_sales.html (Sales risk dashboard)")
    print(f"   - {DB_PATH} (SQLite database)")
    print("\n🚀 Open HTML files in your browser to view interactive dashboards.\n")


if __name__ == "__main__":
    main()


NEXUS DATACOM - POWER MONITORING DASHBOARDS

📦 Step 1: Creating SQLite database...
✅ Database created: nexus_datacom.db
   - Data centers: 27
   - Halls: 113

📊 Step 2: Loading data from database...

📈 Data Summary:
   - Total halls: 113
   - Total projected power: 130,950 kW
   - Average load: 61.1%
   - Average sales: 109.6%

🎨 Step 3: Generating interactive dashboards...
✅ Dashboard saved: heatmap_consumption.html
✅ Dashboard saved: heatmap_sales.html

✅ ALL DASHBOARDS GENERATED SUCCESSFULLY!

📁 Output files:
   - heatmap_consumption.html (Power consumption dashboard)
   - heatmap_sales.html (Sales risk dashboard)
   - nexus_datacom.db (SQLite database)

🚀 Open HTML files in your browser to view interactive dashboards.

